# RAMOSE

RAMOSE v2 reaches both OpenCitations endpoints and joins their results inside one operation. Where the other tools can only join on terms that already coincide in the data, RAMOSE splits an operation into steps and joins the resulting tables on an arbitrary key: here it resolves the DOI to an OMID, then combines the title from Meta with the citation count from Index.

In [1]:
from helper import call

## The join

In [2]:
call("http://localhost:8081/api/v1/citations/10.1007/s11192-022-04367-w?format=json")

curl 'http://localhost:8081/api/v1/citations/10.1007/s11192-022-04367-w?format=json' 

# 200 OK

[
  {
    "doi": "10.1007/s11192-022-04367-w",
    "title": "Identifying And Correcting Invalid Citations Due To DOI Errors In Crossref Data",
    "br": "https://w3id.org/oc/meta/br/061202127149",
    "oc_citation_count": "6"
  }
]


## Output

The same operation in JSON and CSV.

In [3]:
call("http://localhost:8081/api/v1/citations/10.1007/s11192-022-04367-w?format=csv")

curl 'http://localhost:8081/api/v1/citations/10.1007/s11192-022-04367-w?format=csv' 

# 200 OK

doi,title,br,oc_citation_count
10.1007/s11192-022-04367-w,Identifying And Correcting Invalid Citations Due To DOI Errors In Crossref Data,https://w3id.org/oc/meta/br/061202127149,6


## Pagination

A bounded window with RFC 8288 `Link` headers and a known total.

In [4]:
call("http://localhost:8081/api/v1/incoming-citations/10.1007/s11192-022-04367-w?format=json&page=1&page_size=2", show_headers=True)

curl -i 'http://localhost:8081/api/v1/incoming-citations/10.1007/s11192-022-04367-w?format=json&page=1&page_size=2' 



# 200 OK
# Content-Type: application/json
# Link: </api/v1/incoming-citations/10.1007/s11192-022-04367-w?format=json&page=2&page_size=2&total_items=6>; rel="next", </api/v1/incoming-citations/10.1007/s11192-022-04367-w?format=json&page=1&page_size=2&total_items=6>; rel="first", </api/v1/incoming-citations/10.1007/s11192-022-04367-w?format=json&page=3&page_size=2&total_items=6>; rel="last"

[
  {
    "doi": "10.1007/s11192-022-04367-w",
    "br": "https://w3id.org/oc/meta/br/061202127149",
    "citing": "https://w3id.org/oc/meta/br/0609403935"
  },
  {
    "doi": "10.1007/s11192-022-04367-w",
    "br": "https://w3id.org/oc/meta/br/061202127149",
    "citing": "https://w3id.org/oc/meta/br/0609604964"
  }
]


## Versioning

The API version is carried in the base path (`/api/v1`).

## API description

In [5]:
call("http://localhost:8081/api/v1/openapi.yaml")

curl http://localhost:8081/api/v1/openapi.yaml 

# 200 OK

openapi: 3.0.3
info:
  title: OpenCitations Meta + Index (RAMOSE v2 federation demo)
  version: 1.0.0
  description: 'Minimal RAMOSE v2 API over two independent OpenCitations endpoints.
    The `/citations` operation takes a DOI, retrieves the title from OpenCitations
    Meta, and joins the incoming citation count from OpenCitations Index on the shared
    OMID. Nothing in Meta links to Index: the join happens in RAMOSE on the `?br`
    key. The `/incoming-citations` operation reuses the same join to list the citing
    resources one per row, paginated with RFC 8288 `Link` headers.'
  license:
    name: ISC
  contact:
    email: arcangelo.massari@unibo.it
servers:
- url: http://localhost:8081/api/v1
components:
  schemas:
    Error:
      type: object
      properties:
        error:
          type: integer
        message:
          type: string
      required:
      - error
      - message
  parameters:
    require:
      na

## Authentication

Not supported.